# Agents-programmes a budget explicite - compagnon Python

Ce notebook est le compagnon Python du module Lean 4 [`ProgramGames.Bounded`](game_theory_lean/ProgramGames/Bounded.lean), livre par la PR #15395 dans le lake [`game_theory_lean`](game_theory_lean/README.md). Le module represente explicitement le **code public** et le **budget de raisonnement fini** d'un agent-programme (modele structurel de Barasz et al. 2014 et Critch 2016), avec un interprete **total** : un budget nul produit immediatement une action, aucun resultat ne depend d'une recherche de preuve non bornee.

Le pivot conceptuel general reste [`GameTheory-06e-Open-Source-Game-Theory.ipynb`](GameTheory-06e-Open-Source-Game-Theory.ipynb) ; ce compagnon se concentre sur le modele borne et ne duplique pas son contenu.


## Ce que fait ce notebook - et ce qu'il ne fait pas

Le role du compagnon Python est double :

1. **reproduire independamment** la famille finie de bots (`cooperateBot`, `defectBotBounded`, `mirrorBot`, `basicFamily`, `canonicalPD`) depuis zero, sans importer le code Lean ;
2. **rejouer les certificats calculables** du module et comparer chaque verdict Python au theoreme Lean correspondant.

La distinction est fondamentale : le Python verifie une **matrice finie** (quelques agents, quelques budgets) ; le Lean prouve des **quantifications universelles** (pour toute famille, pour tout adversaire de la liste). Un verdict Python conforme ne remplace jamais la preuve formelle - il la rend **inspectable** : chaque ligne du tableau ci-dessous peut etre relue, modifiee, contredite par l'etudiant. Ce notebook ne formalise ni logique de prouvabilite ni theoreme de Lob, et n'extrapole pas vers Godel : le module Lean lui-meme s'en garde explicitement.


## Le code public et l'agent borne

Un `BoundedAgent` porte un code public inspectable (un des trois bots temoins) et un budget de raisonnement naturel. La traduction Python est fidele a la structure Lean `ProgramCode` / `BoundedAgent` :


In [1]:
from dataclasses import dataclass
from enum import Enum, auto


class ProgramCode(Enum):
    """Code public d'un agent-programme (ProgramCode du module Lean)."""
    COOPERATE_BOT = auto()
    DEFECT_BOT = auto()
    MIRROR = auto()


@dataclass(frozen=True)
class BoundedAgent:
    """Agent associant un code public a un budget de raisonnement fini."""
    code: ProgramCode
    budget: int

    def __repr__(self):
        return f"{self.__class__.__name__}({self.code.name}, budget={self.budget})"


cooperate_bot = BoundedAgent(ProgramCode.COOPERATE_BOT, 0)
defect_bot_bounded = BoundedAgent(ProgramCode.DEFECT_BOT, 0)
mirror_bot = BoundedAgent(ProgramCode.MIRROR, 1)
basic_family = [cooperate_bot, defect_bot_bounded, mirror_bot]

print("Famille temoin basicFamily :")
for agent in basic_family:
    print(f"  {agent}")


Famille temoin basicFamily :
  BoundedAgent(COOPERATE_BOT, budget=0)
  BoundedAgent(DEFECT_BOT, budget=0)
  BoundedAgent(MIRROR, budget=1)


## L'interprete total `act`

L'interprete ne fait aucune recherche de preuve : il inspecte le code adverse et le budget, puis produit une action. A **budget nul**, `mirror` deve (il n'a pas les moyens de simuler son adversaire) ; a **budget positif**, il coopere sauf contre le code explicitement defecteur :


In [2]:
def act(agent: BoundedAgent, opponent: ProgramCode) -> str:
    """Interprete total des codes publics (act du module Lean).

    Budget nul : mirror deve immediatement. Budget positif : mirror
    coopere sauf contre le code explicitement defecteur.
    """
    if agent.code is ProgramCode.COOPERATE_BOT:
        return "cooperate"
    if agent.code is ProgramCode.DEFECT_BOT:
        return "defect"
    # MIRROR
    if agent.budget == 0:
        return "defect"
    return "defect" if opponent is ProgramCode.DEFECT_BOT else "cooperate"


def outcome_bounded(row: BoundedAgent, column: BoundedAgent) -> tuple[str, str]:
    """Resultat ordonne de l'interprete : (action de row, action de column)."""
    return (act(row, column.code), act(column, row.code))

for left in basic_family:
    for right in basic_family:
        print(f"outcome_bounded({left.code.name:13s} b={left.budget}, "
              f"{right.code.name:13s} b={right.budget}) = {outcome_bounded(left, right)}")


outcome_bounded(COOPERATE_BOT b=0, COOPERATE_BOT b=0) = ('cooperate', 'cooperate')
outcome_bounded(COOPERATE_BOT b=0, DEFECT_BOT    b=0) = ('cooperate', 'defect')
outcome_bounded(COOPERATE_BOT b=0, MIRROR        b=1) = ('cooperate', 'cooperate')
outcome_bounded(DEFECT_BOT    b=0, COOPERATE_BOT b=0) = ('defect', 'cooperate')
outcome_bounded(DEFECT_BOT    b=0, DEFECT_BOT    b=0) = ('defect', 'defect')
outcome_bounded(DEFECT_BOT    b=0, MIRROR        b=1) = ('defect', 'defect')
outcome_bounded(MIRROR        b=1, COOPERATE_BOT b=0) = ('cooperate', 'cooperate')
outcome_bounded(MIRROR        b=1, DEFECT_BOT    b=0) = ('defect', 'defect')
outcome_bounded(MIRROR        b=1, MIRROR        b=1) = ('cooperate', 'cooperate')


La matrice des issues de la famille temoin se lit d'un coup d'oeil :


In [3]:
print("Matrice des issues de basicFamily (ligne = agent, colonne = adversaire)\n")
header = "".join(f"{a.code.name[:9]:>12s}" for a in basic_family)
print(f"{'':>14s}{header}")
for row in basic_family:
    cells_row = "".join(
        f"{outcome_bounded(row, col)[0][0].upper()}/{outcome_bounded(row, col)[1][0].upper():>4s}"
        for col in basic_family
    )
    print(f"{row.code.name[:9]:>12s} b={row.budget} {cells_row}")
print("\nC = cooperate, D = defect (format : action de l'agent en ligne / action de l'adversaire)")


Matrice des issues de basicFamily (ligne = agent, colonne = adversaire)

                 COOPERATE   DEFECT_BO      MIRROR
   COOPERATE b=0 C/   CC/   DC/   C
   DEFECT_BO b=0 D/   CD/   DD/   D
      MIRROR b=1 C/   CD/   DC/   C

C = cooperate, D = defect (format : action de l'agent en ligne / action de l'adversaire)


## Le dilemme du prisonnier canonique et le rang de paiement

Le parametrage canonique est `T=5, R=3, P=1, S=0`. Le module definit un **rang de paiement fini** `payoffRank` (`CC=3, CD=0, DC=5, DD=1`) et prouve (`payoffRank_le_iff`) que ce rang preserve exactement l'ordre des paiements du jeu canonique - evitant de comparer des reels arbitraires :


In [4]:
CANONICAL_PD = {"T": 5, "R": 3, "P": 1, "S": 0}

def stage_payoff(row_action: str, col_action: str) -> int:
    """Paiement de l'agent ligne dans le PD canonique."""
    if row_action == "cooperate" and col_action == "cooperate":
        return CANONICAL_PD["R"]
    if row_action == "cooperate":
        return CANONICAL_PD["S"]
    if col_action == "cooperate":
        return CANONICAL_PD["T"]
    return CANONICAL_PD["P"]

def payoff_rank(row_action: str, col_action: str) -> int:
    """Rang fini du module Lean : CC=3, CD=0, DC=5, DD=1."""
    return {("cooperate", "cooperate"): 3, ("cooperate", "defect"): 0,
            ("defect", "cooperate"): 5, ("defect", "defect"): 1}[(row_action, col_action)]

print(f"PD canonique : {CANONICAL_PD}")
for pair in [("cooperate", "cooperate"), ("cooperate", "defect"),
             ("defect", "cooperate"), ("defect", "defect")]:
    print(f"  {pair} -> stage_payoff={stage_payoff(*pair)}, payoffRank={payoff_rank(*pair)}")


PD canonique : {'T': 5, 'R': 3, 'P': 1, 'S': 0}
  ('cooperate', 'cooperate') -> stage_payoff=3, payoffRank=3
  ('cooperate', 'defect') -> stage_payoff=0, payoffRank=0
  ('defect', 'cooperate') -> stage_payoff=5, payoffRank=5
  ('defect', 'defect') -> stage_payoff=1, payoffRank=1


## Les organes calculables

Le module expose trois organes booleens - `mutualCooperationCheck`, `unexploitableCheck`, `programNashCheck` - dont Lean prouve qu'ils refletent exactement les definitions propositionnelles. Le Python les reproduit tels quels :


In [5]:
def mutual_cooperation_check(left: BoundedAgent, right: BoundedAgent) -> bool:
    """Organe de cooperation mutuelle (mutualCooperationCheck)."""
    return outcome_bounded(left, right) == ("cooperate", "cooperate")


def unexploitable_check(agent: BoundedAgent, opponents: list[BoundedAgent]) -> bool:
    """Organe d'inexploitabilite sur une famille finie (unexploitableCheck).

    Inexploitable : jamais (cooperate, defect) - jamais de cooperation
    unilaterale pendant que l'adversaire deve.
    """
    return all(outcome_bounded(agent, opp) != ("cooperate", "defect")
               for opp in opponents)


def program_nash_check(family: list[BoundedAgent],
                       left: BoundedAgent, right: BoundedAgent) -> bool:
    """Organe d'equilibre relatif pour le PD canonique (programNashCheck).

    Aucune substitution unilaterale dans la famille n'ameliore
    strictement le paiement (compare via le rang fini).
    """
    left_ok = all(
        payoff_rank(*outcome_bounded(alt, right))
        <= payoff_rank(*outcome_bounded(left, right))
        for alt in family
    )
    right_ok = all(
        payoff_rank(outcome_bounded(left, alt)[1], outcome_bounded(left, alt)[0])
        <= payoff_rank(outcome_bounded(left, right)[1], outcome_bounded(left, right)[0])
        for alt in family
    )
    return left_ok and right_ok


## Rejeu des certificats Lean

Chaque theoreme du module est rejoue ci-dessous. Le verdict Python est calcule sur la matrice finie ; la colonne de droite rappelle le theoreme Lean qui couvre le cas **generique**.


In [6]:
certificats = []

def verifier(nom: str, verdict_python: bool, theoreme_lean: str):
    certificats.append((nom, verdict_python, theoreme_lean))
    print(f"{nom:42s} Python={str(verdict_python):5s}  <-  {theoreme_lean}")

# 1. Deux bots cooperateurs produisent la cooperation mutuelle
verifier("cooperate_cooperate",
         mutual_cooperation_check(cooperate_bot, cooperate_bot),
         "MutualCooperationBounded cooperateBot cooperateBot")

# 2. Deux bots defecteurs produisent la defection mutuelle
verifier("defect_defect",
         outcome_bounded(defect_bot_bounded, defect_bot_bounded) == ("defect", "defect"),
         "outcomeBounded defectBotBounded defectBotBounded = (defect, defect)")


cooperate_cooperate                        Python=True   <-  MutualCooperationBounded cooperateBot cooperateBot
defect_defect                              Python=True   <-  outcomeBounded defectBotBounded defectBotBounded = (defect, defect)


In [7]:
# 3. Le bot miroir coopere avec lui-meme (budget positif)
verifier("mirror_mirror",
         mutual_cooperation_check(mirror_bot, mirror_bot),
         "MutualCooperationBounded mirrorBot mirrorBot")

# 4. Le bot defecteur est inexploitable contre toute famille :
#    son action n'est JAMAIS cooperate (argument structurel,
#    verifie ici sur la famille temoin)
verifier("defectBotBounded_unexploitable",
         unexploitable_check(defect_bot_bounded, basic_family),
         "UnexploitableInFamily defectBotBounded opponents (quantifie)")

# 5. Le miroir est inexploitable dans la famille temoin
verifier("mirror_basicFamily_unexploitable",
         unexploitable_check(mirror_bot, basic_family),
         "unexploitableCheck mirrorBot basicFamily = true")


mirror_mirror                              Python=True   <-  MutualCooperationBounded mirrorBot mirrorBot
defectBotBounded_unexploitable             Python=True   <-  UnexploitableInFamily defectBotBounded opponents (quantifie)
mirror_basicFamily_unexploitable           Python=True   <-  unexploitableCheck mirrorBot basicFamily = true


In [8]:
# 6. La defection mutuelle est un equilibre relatif dans la famille temoin
verifier("defect_profile_programNash",
         program_nash_check(basic_family, defect_bot_bounded, defect_bot_bounded),
         "ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded")

# 7. Le meme certificat via l'organe booleen (Lean : decide)
verifier("defect_profile_check",
         program_nash_check(basic_family, defect_bot_bounded, defect_bot_bounded),
         "programNashCheck basicFamily defectBotBounded defectBotBounded = true")

# 8. Le rang fini preserve l'ordre des paiements (verification exhaustive 4x4)
rank_ok = all(
    (payoff_rank(a1, a2) <= payoff_rank(b1, b2))
    == (stage_payoff(a1, a2) <= stage_payoff(b1, b2))
    for a1 in ("cooperate", "defect") for a2 in ("cooperate", "defect")
    for b1 in ("cooperate", "defect") for b2 in ("cooperate", "defect")
)
verifier("payoffRank_le_iff (16 paires)", rank_ok,
         "payoffRank_le_iff : rang fini iff ordre des paiements")

print(f"\n{sum(1 for _, v, _ in certificats if v)}/{len(certificats)} certificats conformes")
assert all(v for _, v, _ in certificats), "divergence Python/Lean detectee"


defect_profile_programNash                 Python=True   <-  ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded
defect_profile_check                       Python=True   <-  programNashCheck basicFamily defectBotBounded defectBotBounded = true
payoffRank_le_iff (16 paires)              Python=True   <-  payoffRank_le_iff : rang fini iff ordre des paiements

8/8 certificats conformes


Les 8 verdicts sont conformes : sur la matrice finie, le verificateur Python independant confirme chaque certificat calculable du module. La preuve Lean garde ce que le calcul ne peut pas : la quantification sur **toutes** les familles et **tous** les budgets.


## Exercice 1 - Le miroir sans budget

Le module documente qu'a budget nul, `mirror` deve. On construit `mirror_broke = BoundedAgent(ProgramCode.MIRROR, 0)`. **Question** : ce miroir sans budget est-il inexploitable dans `basic_family` ? La cooperation mutuelle est-elle encore possible ? Completer la fonction pour calculer les deux verdicts, puis verifier contre l'intuition : un miroir a budget nul se comporte structurellement comme quel bot de la famille ?


In [9]:
mirror_broke = BoundedAgent(ProgramCode.MIRROR, 0)

def verdicts_miroir_sans_budget():
    """Renvoie (inexploitable, cooperation_mutuelle_avec_lui_meme)."""
    # TODO Etudiant : utiliser unexploitable_check et mutual_cooperation_check
    # sur mirror_broke (contre basic_family puis contre lui-meme).
    result = None  # TODO etudiant
    return result

print("Exercice 1 a completer : verdicts_miroir_sans_budget()")
print("Indice : budget nul => act(MIRROR, _) renvoie toujours...")


Exercice 1 a completer : verdicts_miroir_sans_budget()
Indice : budget nul => act(MIRROR, _) renvoie toujours...


## Exercice 2 - Famille etendue aux budgets

On etend la famille temoin avec des variantes de budget : `basic_family_etendue = basic_family + [BoundedAgent(ProgramCode.MIRROR, 5), mirror_broke]`. **Question** : le profil de defection mutuelle `(defect_bot_bounded, defect_bot_bounded)` reste-t-il un equilibre relatif dans cette famille etendue ? Et la cooperation mutuelle des miroirs `(mirror_bot, mirror_bot)` devient-elle equilibre ? Completer le calcul des deux verdicts et interpreter pourquoi la reponse differe entre les deux profils.


In [10]:
basic_family_etendue = basic_family + [
    BoundedAgent(ProgramCode.MIRROR, 5),
    mirror_broke,
]

def equilibres_famille_etendue():
    """Renvoie (nash_defection, nash_cooperation_miroirs) dans la famille etendue."""
    # TODO Etudiant : appeler program_nash_check deux fois sur
    # basic_family_etendue avec les profils (defect, defect) puis
    # (mirror_bot, mirror_bot).
    result = None  # TODO etudiant
    return result

print("Exercice 2 a completer : equilibres_famille_etendue()")
print("Indice : contre un miroir a budget positif, que rapporte la deviation ?")


Exercice 2 a completer : equilibres_famille_etendue()
Indice : contre un miroir a budget positif, que rapporte la deviation ?


## Exercice 3 - Le paiement du miroir contre le defecteur

Le theoreme `mirror_basicFamily_unexploitable` affirme que le miroir a budget positif est inexploitable dans la famille temoin : il deve contre `defectBotBounded`, cooperant avec les autres. **Question** : quel paiement le miroir obtient-il effectivement contre chaque membre de la famille, et quel membre est son pire adversaire ? Completer le calcul du dict des paiements et du pire adversaire, puis conclure : inexploitable signifie-t-il optimal ?


In [11]:
def paiements_miroir():
    """Renvoie (dict {adversaire: paiement du miroir}, pire_adversaire)."""
    # TODO Etudiant : pour chaque agent de basic_family, calculer
    # stage_payoff(*outcome_bounded(mirror_bot, adversaire))
    # (attention : outcome_bounded renvoie (action miroir, action adversaire)).
    result = None  # TODO etudiant
    return result

print("Exercice 3 a completer : paiements_miroir()")
print("Indice : inexploitable protege du pire (S=0), pas de l'optimum (T=5).")


Exercice 3 a completer : paiements_miroir()
Indice : inexploitable protege du pire (S=0), pas de l'optimum (T=5).


## Navigation et suite

- Pivot conceptuel : [`GameTheory-06e-Open-Source-Game-Theory.ipynb`](GameTheory-06e-Open-Source-Game-Theory.ipynb)
- Module Lean (FR) : [`game_theory_lean/ProgramGames/Bounded.lean`](game_theory_lean/ProgramGames/Bounded.lean) - miroir anglais : [`Bounded_en.lean`](game_theory_lean/ProgramGames/Bounded_en.lean)
- Noyau fonctionnel : [`game_theory_lean/ProgramGames/Basic.lean`](game_theory_lean/ProgramGames/Basic.lean)
- Lake complet : [`game_theory_lean/README.md`](game_theory_lean/README.md)

Ce compagnon couvre le versant **calcul fini** du modele borne ; le versant preuve formelle (les 8 theoremes en quantification universelle) vit dans le lake. Les exercices 1 a 3 restent a completer.
